In [1]:
import numpy as np
import pandas as pd
import os, vcf, sys, itertools, shutil, subprocess, glob, yaml
from Bio import Entrez, Seq, SeqIO
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
import matplotlib.pyplot as plt

isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")
genomic_data_dir = "/n/data1/hms/dbmi/farhat/rollingDB/genomic_data"

TRUST_isolate_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/TRUST/isolate_variants_WHO_catalog.csv")

In [2]:
TRUST_samples = os.listdir("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/Illumina_culture_WGS_processed")
len(TRUST_samples)

782

In [3]:
TRUST_finished_samples = [os.path.basename(fName).split('_')[0] for fName in glob.glob("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/Illumina_culture_WGS_processed/*/pilon/*_variants.vcf")]

len(TRUST_finished_samples)

753

In [4]:
TRUST_isolate_variants.query("drug=='Isoniazid'").GENE.unique()

array(['Rv1258c', 'mshA', 'katG', 'ndh', 'dnaA', 'Rv0010c', "oxyR'-ahpC",
       'glpK', 'hadA', 'Rv1482c-fabG1', 'ahpC', 'Rv0010c-Rv0011c', 'inhA',
       'Rv1129c', 'Rv2752c', 'fabG1'], dtype=object)

In [5]:
isoniazid_genes = ['katG', "oxyR'-ahpC", 'inhA', 'Rv1482c-fabG1', 'ahpC', 'fabG1']

In [6]:
# previous version sent to Marie. Should have been named 20241205 though
df = pd.read_csv("./INH_variants_TRUST_20251205.csv")

# current mapping
TRUST_pid_WGS_mapping = pd.read_csv("~/TRUST_data_processing/processed_data/20250520_combined_patient_WGS_data.csv")

# Get only variants in the Tier 1 Isoniazid Genes (the Tier 2 do not confer resistance and are hard to interpret)

In [7]:
TRUST_INH_variants = TRUST_isolate_variants.rename(columns={'ROLLINGDB_ID': 'SampleID'}).loc[(TRUST_isolate_variants['drug'].isin(['Isoniazid', np.nan])) & (TRUST_isolate_variants['GENE'].isin(isoniazid_genes))].merge(df[['pid', 'Original_ID', 'SampleID']])

# drop duplicates that are the same row but have different names. The names not in the catalogue will be NA, so put NAs last
TRUST_INH_variants = TRUST_INH_variants.sort_values(['confidence'], na_position='last').drop_duplicates(['pid', 'POS', 'REF', 'ALT'], keep='first')

TRUST_INH_variants = TRUST_INH_variants.reset_index(drop=True)

TRUST_INH_variants['confidence'] = TRUST_INH_variants['confidence'].fillna('Not in Catalogue')

TRUST_INH_variants['Count'] = 1

# Pivot to Matrix

In [8]:
TRUST_INH_variant_matrix = TRUST_INH_variants.pivot(index='pid', columns=['confidence', 'variant'], values='Count').fillna(0).astype(int)

TRUST_INH_variant_matrix = TRUST_INH_variant_matrix.sort_index(axis=1, level=[0, 1], ascending=[True, True])

# Add back in pids without WGS

In [9]:
add_pids = list(set(df.pid) - set(TRUST_INH_variant_matrix.index))

# these pids have sequencing but no INH related variants
pids_no_variant = TRUST_pid_WGS_mapping.query("pid in @add_pids").pid.unique()

pids_no_WGS = list(set(add_pids) - set(pids_no_variant))

len(pids_no_variant), len(pids_no_WGS)

(155, 35)

In [10]:
pids_no_variant_df = pd.DataFrame(0, index=pids_no_variant, columns = TRUST_INH_variant_matrix.columns)
pids_no_WGS_df = pd.DataFrame(np.nan, index=pids_no_WGS, columns = TRUST_INH_variant_matrix.columns)

# Check pids that newly have WGS

In [11]:
set(pd.concat([TRUST_INH_variant_matrix, pids_no_variant_df]).index) - set(df.query("WGS==1").pid)

{'T0148', 'T0434', 'T0450'}

In [12]:
set(df.query("WGS==0").pid) - set(pids_no_WGS_df.index)

{'T0148', 'T0434', 'T0450'}

# Combine into a single matrix

In [13]:
TRUST_INH_variant_matrix = pd.concat([TRUST_INH_variant_matrix, pids_no_variant_df, pids_no_WGS_df])
TRUST_INH_variant_matrix.to_csv("./INH_variant_matrix_for_Marie.csv")

# Subset to resistance-associated variants

In [16]:
TRUST_INH_R_variant_matrix = TRUST_INH_variant_matrix.loc[:, TRUST_INH_variant_matrix.columns.get_level_values(0).str.contains('Assoc w R')]
TRUST_INH_R_variant_matrix.to_csv("./INH_resistance_variant_matrix_for_Marie.csv")

# Get total counts

In [15]:
INH_variant_count_aggregated = pd.DataFrame(TRUST_INH_variant_matrix.sum(axis=0)).reset_index().rename(columns={0: 'Count'})
INH_variant_count_aggregated.to_csv("./INH_variant_count_aggregated.csv", index=False)